In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import joblib
import os

In [4]:
# Load dataset

df = pd.read_csv("../data/raw/supervised_dataset.csv")

print("Initial Shape:", df.shape)
print("Missing Values:", df.isnull().sum().sum())

Initial Shape: (1699, 12)
Missing Values: 8


In [5]:
df = df.drop(columns=["_id", "Unnamed: 0"], errors="ignore")

In [6]:
duplicate_count = df.duplicated().sum()

print("Duplicate Rows:", duplicate_count)

Duplicate Rows: 37


In [8]:
duplicates = df[df.duplicated(keep=False)]

print(duplicates)

      inter_api_access_duration(sec)  api_access_uniqueness  \
1117                        0.002000               1.000000   
1119                        0.001782               0.057762   
1120                        0.001782               0.057762   
1121                        0.001782               0.057762   
1122                        0.001782               0.057762   
1140                        0.008000               0.500000   
1179                        0.011000               1.000000   
1182                       11.721073               0.323077   
1183                       11.721073               0.323077   
1184                       11.721073               0.323077   
1185                       11.721073               0.323077   
1190                        0.011000               1.000000   
1192                        0.480756               0.511628   
1193                        0.480756               0.511628   
1194                        0.480756               0.51

In [11]:
df = df.drop_duplicates().reset_index(drop=True)

In [13]:
X = df.drop("classification", axis=1)

y = df["classification"]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print("Classes:", np.unique(y))

Classes: [0 1]


In [14]:
categorical_cols = ["ip_type", "source"]

numerical_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)

Numerical: ['inter_api_access_duration(sec)', 'api_access_uniqueness', 'sequence_length(count)', 'vsession_duration(min)', 'num_sessions', 'num_users', 'num_unique_apis']
Categorical: ['ip_type', 'source']


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Reset indexes
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = pd.Series(y_train).reset_index(drop=True)
y_test = pd.Series(y_test).reset_index(drop=True)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (1329, 9)
Test Shape: (333, 9)


In [18]:
common = pd.merge(X_train, X_test, how='inner')

print("Common Rows:", len(common))

Common Rows: 0


In [21]:
# Use absolute paths to avoid permission issues
PROCESSED_PATH = r"c:\Users\teler\Desktop\api-security-system\ml-service\data\processed"
MODEL_PATH = r"c:\Users\teler\Desktop\api-security-system\ml-service\model"

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

X_train.to_csv(os.path.join(PROCESSED_PATH, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(PROCESSED_PATH, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(PROCESSED_PATH, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(PROCESSED_PATH, "y_test.csv"), index=False)

joblib.dump(label_encoder, os.path.join(MODEL_PATH, "label_encoder.pkl"))

print("Saved Successfully")

Saved Successfully
